In [ ]:
import os
import re
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
import plotly.io as pio
from typing import Dict, List, Optional

pio.renderers.default = "plotly_mimetype"

# CC mode mapping (from ns3 config file)
CC_MODES = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}

def parse_config_sum(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        pass
    return params

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_config_file(folder_path: str) -> Optional[str]:
    """Finds a .txt configuration file in the 'configs' subfolder of a run directory."""
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0

def add_bar_with_errors(fig, x, y_mean, y_min, y_max, name, color, error_color):
    """Adds a grouped bar trace with asymmetric min/max error bars."""
    fig.add_trace(go.Bar(
        x=x, y=y_mean, name=name,
        marker_color=color,
        error_y=dict(
            type='data', symmetric=False,
            array=y_max - y_mean,
            arrayminus=y_mean - y_min,
            color=error_color, thickness=1.5, width=4
        )
    ))

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
TOPOLOGIES = ['experiment3/FoldedClosECMP']
WORKLOAD_GROUP = 'Llama8B'

all_results = []
ns3_variant_counts = {}   # diagnostics: how many runs per variant

for topo in TOPOLOGIES:
    topo_path = os.path.join(BASE_OUTPUT_DIR, topo, WORKLOAD_GROUP)
    if not os.path.isdir(topo_path):
        print(f"Directory not found for topology {topo}, skipping.")
        continue

    for workload_name in os.listdir(topo_path):
        workload_path = os.path.join(topo_path, workload_name)
        if not os.path.isdir(workload_path):
            continue

        workload_results = {'workload': workload_name, 'topology': topo}

        ns3_dctcp_exec_times, ns3_dctcp_sim_times = [], []
        ns3_dcqcn_exec_times, ns3_dcqcn_sim_times = [], []
        g2_exec_times, g2_sim_times = [], []

        run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]

        for run_dir_name in run_dirs:
            run_path = os.path.join(workload_path, run_dir_name)

            sim_type = None
            if os.path.isdir(os.path.join(run_path, 'g2')):
                sim_type = 'g2'
            elif os.path.isdir(os.path.join(run_path, 'ns3')):
                sim_type = 'ns3'
            elif os.path.isdir(os.path.join(run_path, 'analytical_unaware')):
                sim_type = 'analytical_unaware'

            if not sim_type:
                continue

            sim_path = os.path.join(run_path, sim_type)

            # For NS3: detect protocol via cc_mode in the configs/ subfolder
            ns3_variant = None
            if sim_type == 'ns3':
                config_file = find_config_file(run_path)
                if config_file:
                    ns3_params = parse_config(config_file)
                    cc_mode_val = ns3_params.get('cc_mode', '')
                    try:
                        cc_mode_int = int(cc_mode_val)
                    except (ValueError, TypeError):
                        cc_mode_int = -1
                    if cc_mode_int == 8:
                        ns3_variant = 'DCTCP'
                    elif cc_mode_int == 1:
                        ns3_variant = 'DCQCN'
                    else:
                        ns3_variant = CC_MODES.get(cc_mode_int, f'CC{cc_mode_int}')
                else:
                    # Fallback: try to infer from run_summary or folder name
                    summary_p = parse_config_sum(os.path.join(run_path, 'run_summary.txt'))
                    ns3_variant = summary_p.get('cc mode', summary_p.get('protocol', 'NS3_unknown'))

                ns3_variant_counts[ns3_variant] = ns3_variant_counts.get(ns3_variant, 0) + 1

            # Extract Estimated Execution Time
            timing_file = next(
                (os.path.join(sim_path, f) for f in os.listdir(sim_path) if 'trace_matched_timing.csv' in f),
                None
            )
            max_time_ns = None
            if timing_file:
                try:
                    df_timing = pd.read_csv(timing_file)
                    if 'callback_tick' in df_timing.columns:
                        max_time_ns = df_timing['callback_tick'].max()
                except Exception as e:
                    print(f"Error reading {timing_file}: {e}")

            # Extract Simulation Wall-Clock Time
            summary_params = parse_config_sum(os.path.join(run_path, 'run_summary.txt'))
            sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))

            if sim_type == 'g2':
                if max_time_ns is not None: g2_exec_times.append(max_time_ns)
                if sim_time_sec is not None: g2_sim_times.append(sim_time_sec)
            elif sim_type == 'ns3' and ns3_variant == 'DCTCP':
                if max_time_ns is not None: ns3_dctcp_exec_times.append(max_time_ns)
                if sim_time_sec is not None: ns3_dctcp_sim_times.append(sim_time_sec)
            elif sim_type == 'ns3' and ns3_variant == 'DCQCN':
                if max_time_ns is not None: ns3_dcqcn_exec_times.append(max_time_ns)
                if sim_time_sec is not None: ns3_dcqcn_sim_times.append(sim_time_sec)
            elif sim_type == 'analytical_unaware':
                workload_results['Analytical_Est_Exec_Time_ns'] = max_time_ns
                workload_results['Analytical_Sim_Time_sec'] = sim_time_sec

        # G2 stats
        if g2_exec_times:
            workload_results['G2_Est_Exec_Time_ns']     = sum(g2_exec_times) / len(g2_exec_times)
            workload_results['G2_Est_Exec_Time_ns_min'] = min(g2_exec_times)
            workload_results['G2_Est_Exec_Time_ns_max'] = max(g2_exec_times)
        if g2_sim_times:
            workload_results['G2_Sim_Time_sec']     = sum(g2_sim_times) / len(g2_sim_times)
            workload_results['G2_Sim_Time_sec_min'] = min(g2_sim_times)
            workload_results['G2_Sim_Time_sec_max'] = max(g2_sim_times)

        # NS3 DCTCP stats
        if ns3_dctcp_exec_times:
            workload_results['NS3_DCTCP_Est_Exec_Time_ns']     = sum(ns3_dctcp_exec_times) / len(ns3_dctcp_exec_times)
            workload_results['NS3_DCTCP_Est_Exec_Time_ns_min'] = min(ns3_dctcp_exec_times)
            workload_results['NS3_DCTCP_Est_Exec_Time_ns_max'] = max(ns3_dctcp_exec_times)
        if ns3_dctcp_sim_times:
            workload_results['NS3_DCTCP_Sim_Time_sec']     = sum(ns3_dctcp_sim_times) / len(ns3_dctcp_sim_times)
            workload_results['NS3_DCTCP_Sim_Time_sec_min'] = min(ns3_dctcp_sim_times)
            workload_results['NS3_DCTCP_Sim_Time_sec_max'] = max(ns3_dctcp_sim_times)

        # NS3 DCQCN stats
        if ns3_dcqcn_exec_times:
            workload_results['NS3_DCQCN_Est_Exec_Time_ns']     = sum(ns3_dcqcn_exec_times) / len(ns3_dcqcn_exec_times)
            workload_results['NS3_DCQCN_Est_Exec_Time_ns_min'] = min(ns3_dcqcn_exec_times)
            workload_results['NS3_DCQCN_Est_Exec_Time_ns_max'] = max(ns3_dcqcn_exec_times)
        if ns3_dcqcn_sim_times:
            workload_results['NS3_DCQCN_Sim_Time_sec']     = sum(ns3_dcqcn_sim_times) / len(ns3_dcqcn_sim_times)
            workload_results['NS3_DCQCN_Sim_Time_sec_min'] = min(ns3_dcqcn_sim_times)
            workload_results['NS3_DCQCN_Sim_Time_sec_max'] = max(ns3_dcqcn_sim_times)

        if len(workload_results) > 2:
            all_results.append(workload_results)

# Diagnostics
print(f"NS3 variant detection counts: {ns3_variant_counts}")
print(f"Total workload entries collected: {len(all_results)}")
if all_results:
    sample_keys = list(all_results[0].keys())
    print(f"Keys in first result: {sample_keys}")

# ---------------------------------------------------------------------------
# Build DataFrame, compute errors / speedups, generate plots & tables
# ---------------------------------------------------------------------------
if all_results:
    df = pd.DataFrame(all_results)

    # Only require columns that actually exist in the DataFrame
    desired_required = [
        'G2_Est_Exec_Time_ns', 'G2_Est_Exec_Time_ns_min', 'G2_Est_Exec_Time_ns_max',
        'NS3_DCTCP_Est_Exec_Time_ns', 'NS3_DCTCP_Est_Exec_Time_ns_min', 'NS3_DCTCP_Est_Exec_Time_ns_max',
        'NS3_DCQCN_Est_Exec_Time_ns', 'NS3_DCQCN_Est_Exec_Time_ns_min', 'NS3_DCQCN_Est_Exec_Time_ns_max',
        'Analytical_Est_Exec_Time_ns',
        'G2_Sim_Time_sec', 'G2_Sim_Time_sec_min', 'G2_Sim_Time_sec_max',
        'NS3_DCTCP_Sim_Time_sec', 'NS3_DCTCP_Sim_Time_sec_min', 'NS3_DCTCP_Sim_Time_sec_max',
        'NS3_DCQCN_Sim_Time_sec', 'NS3_DCQCN_Sim_Time_sec_min', 'NS3_DCQCN_Sim_Time_sec_max',
        'Analytical_Sim_Time_sec',
    ]
    required_cols = [c for c in desired_required if c in df.columns]
    missing_cols = [c for c in desired_required if c not in df.columns]
    if missing_cols:
        print(f"WARNING: Missing columns (no data found for): {missing_cols}")

    df.dropna(subset=required_cols, inplace=True)
    df.sort_values(by=['topology', 'workload'], inplace=True)

    # Determine which NS3 variants are available
    has_dctcp = 'NS3_DCTCP_Est_Exec_Time_ns' in df.columns
    has_dcqcn = 'NS3_DCQCN_Est_Exec_Time_ns' in df.columns

    # Errors (signed %)
    if has_dctcp:
        df['G2 Error vs DCTCP (%)'] = ((df['G2_Est_Exec_Time_ns'] - df['NS3_DCTCP_Est_Exec_Time_ns']) / df['NS3_DCTCP_Est_Exec_Time_ns']) * 100
        df['AU Error vs DCTCP (%)'] = ((df['Analytical_Est_Exec_Time_ns'] - df['NS3_DCTCP_Est_Exec_Time_ns']) / df['NS3_DCTCP_Est_Exec_Time_ns']) * 100
        df['G2 Speedup vs DCTCP (x)'] = df['NS3_DCTCP_Sim_Time_sec'] / df['G2_Sim_Time_sec']
        df['AU Speedup vs DCTCP (x)'] = df['NS3_DCTCP_Sim_Time_sec'] / df['Analytical_Sim_Time_sec']
    if has_dcqcn:
        df['G2 Error vs DCQCN (%)'] = ((df['G2_Est_Exec_Time_ns'] - df['NS3_DCQCN_Est_Exec_Time_ns']) / df['NS3_DCQCN_Est_Exec_Time_ns']) * 100
        df['AU Error vs DCQCN (%)'] = ((df['Analytical_Est_Exec_Time_ns'] - df['NS3_DCQCN_Est_Exec_Time_ns']) / df['NS3_DCQCN_Est_Exec_Time_ns']) * 100
        df['G2 Speedup vs DCQCN (x)'] = df['NS3_DCQCN_Sim_Time_sec'] / df['G2_Sim_Time_sec']
        df['AU Speedup vs DCQCN (x)'] = df['NS3_DCQCN_Sim_Time_sec'] / df['Analytical_Sim_Time_sec']

    print(f"\n\n{'='*80}\n--- Generated LaTeX Tables ---\n{'='*80}")

    for topo in df['topology'].unique():
        topo_df = df[df['topology'] == topo].copy()
        if topo_df.empty:
            continue

        sort_col = 'NS3_DCTCP_Est_Exec_Time_ns' if has_dctcp else ('NS3_DCQCN_Est_Exec_Time_ns' if has_dcqcn else 'G2_Est_Exec_Time_ns')
        topo_df = topo_df.sort_values(by=sort_col).reset_index(drop=True)

        topo_df['short_workload'] = (
            topo_df['workload']
            .str.replace('Llama8B_last_', '', regex=False)
            .str.replace('.seq_2048.batch_64', '', regex=False)
            .str.replace('_', '-', regex=False)
            .str.replace('.seq-2048.batch-1024', '', regex=False)
        )

        print(f"\n{'='*80}\n--- Results for Topology: {topo} ---\n{'='*80}")

        # ---------------------------------------------------------------
        # Plot 1: Absolute Estimated Execution Time (up to 4 bars)
        # ---------------------------------------------------------------
        fig_abs = go.Figure()

        if has_dctcp:
            add_bar_with_errors(fig_abs,
                topo_df['short_workload'],
                topo_df['NS3_DCTCP_Est_Exec_Time_ns'],
                topo_df['NS3_DCTCP_Est_Exec_Time_ns_min'],
                topo_df['NS3_DCTCP_Est_Exec_Time_ns_max'],
                'NS3 DCTCP', 'lightgray', 'gray')

        if has_dcqcn:
            add_bar_with_errors(fig_abs,
                topo_df['short_workload'],
                topo_df['NS3_DCQCN_Est_Exec_Time_ns'],
                topo_df['NS3_DCQCN_Est_Exec_Time_ns_min'],
                topo_df['NS3_DCQCN_Est_Exec_Time_ns_max'],
                'NS3 DCQCN', 'darkgray', 'black')

        add_bar_with_errors(fig_abs,
            topo_df['short_workload'],
            topo_df['G2_Est_Exec_Time_ns'],
            topo_df['G2_Est_Exec_Time_ns_min'],
            topo_df['G2_Est_Exec_Time_ns_max'],
            'G2', 'skyblue', 'blue')

        fig_abs.add_trace(go.Bar(
            x=topo_df['short_workload'],
            y=topo_df['Analytical_Est_Exec_Time_ns'],
            name='Analytical', marker_color='salmon'
        ))

        fig_abs.update_layout(
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Estimated Execution Time (ns)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        fig_abs.show()

        # ---------------------------------------------------------------
        # Plot 2: Relative Execution Time (normalised to NS3 DCTCP = 100%)
        # ---------------------------------------------------------------
        ref_col = 'NS3_DCTCP_Est_Exec_Time_ns' if has_dctcp else 'NS3_DCQCN_Est_Exec_Time_ns'
        ref_label = 'NS3 DCTCP' if has_dctcp else 'NS3 DCQCN'
        ref_min_col = ref_col.replace('_Est_Exec_Time_ns', '_Est_Exec_Time_ns_min')
        ref_max_col = ref_col.replace('_Est_Exec_Time_ns', '_Est_Exec_Time_ns_max')

        fig_norm = go.Figure()

        # Reference bar (DCTCP or DCQCN) at 100%
        ref_norm_min = (topo_df[ref_min_col] / topo_df[ref_col]) * 100
        ref_norm_max = (topo_df[ref_max_col] / topo_df[ref_col]) * 100
        fig_norm.add_trace(go.Bar(
            x=topo_df['short_workload'], y=[100] * len(topo_df),
            name=ref_label, marker_color='lightgray',
            error_y=dict(type='data', symmetric=False,
                         array=ref_norm_max - 100, arrayminus=100 - ref_norm_min,
                         color='gray', thickness=1.5, width=4)
        ))

        # Second NS3 variant normalised to the reference
        if has_dctcp and has_dcqcn:
            dcqcn_norm_mean = (topo_df['NS3_DCQCN_Est_Exec_Time_ns']     / topo_df[ref_col]) * 100
            dcqcn_norm_min  = (topo_df['NS3_DCQCN_Est_Exec_Time_ns_min'] / topo_df[ref_col]) * 100
            dcqcn_norm_max  = (topo_df['NS3_DCQCN_Est_Exec_Time_ns_max'] / topo_df[ref_col]) * 100
            fig_norm.add_trace(go.Bar(
                x=topo_df['short_workload'], y=dcqcn_norm_mean,
                name='NS3 DCQCN', marker_color='darkgray',
                error_y=dict(type='data', symmetric=False,
                             array=dcqcn_norm_max - dcqcn_norm_mean,
                             arrayminus=dcqcn_norm_mean - dcqcn_norm_min,
                             color='black', thickness=1.5, width=4)
            ))

        g2_norm_mean = (topo_df['G2_Est_Exec_Time_ns']     / topo_df[ref_col]) * 100
        g2_norm_min  = (topo_df['G2_Est_Exec_Time_ns_min'] / topo_df[ref_col]) * 100
        g2_norm_max  = (topo_df['G2_Est_Exec_Time_ns_max'] / topo_df[ref_col]) * 100
        fig_norm.add_trace(go.Bar(
            x=topo_df['short_workload'], y=g2_norm_mean,
            name='G2', marker_color='skyblue',
            error_y=dict(type='data', symmetric=False,
                         array=g2_norm_max - g2_norm_mean,
                         arrayminus=g2_norm_mean - g2_norm_min,
                         color='blue', thickness=1.5, width=4)
        ))

        au_norm = (topo_df['Analytical_Est_Exec_Time_ns'] / topo_df[ref_col]) * 100
        fig_norm.add_trace(go.Bar(
            x=topo_df['short_workload'], y=au_norm,
            name='Analytical', marker_color='salmon'
        ))

        fig_norm.update_layout(
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title=f'Relative Execution Time to {ref_label} (%)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        fig_norm.show()

        # ---------------------------------------------------------------
        # Spearman's Rank Correlation
        # ---------------------------------------------------------------
        print(f"\n--- Spearman's Rank Correlation for {topo} ---")
        for gt_label, ns3_col in ([('DCTCP', 'NS3_DCTCP_Est_Exec_Time_ns')] if has_dctcp else []) + \
                                  ([('DCQCN', 'NS3_DCQCN_Est_Exec_Time_ns')] if has_dcqcn else []):
            g2_corr, _ = spearmanr(topo_df[ns3_col], topo_df['G2_Est_Exec_Time_ns'])
            au_corr, _ = spearmanr(topo_df[ns3_col], topo_df['Analytical_Est_Exec_Time_ns'])
            print(f"  G2 vs NS3 {gt_label}:         {g2_corr:.4f}")
            print(f"  Analytical vs NS3 {gt_label}: {au_corr:.4f}")

        # ---------------------------------------------------------------
        # Inline Summary Table
        # ---------------------------------------------------------------
        summary_dict = {'Workload': topo_df['short_workload'].values}
        fmt_dict = {}
        if has_dctcp:
            summary_dict['NS3 DCTCP (ns)']   = topo_df['NS3_DCTCP_Est_Exec_Time_ns'].values
            summary_dict['G2 Err DCTCP (%)'] = topo_df['G2 Error vs DCTCP (%)'].values
            summary_dict['AU Err DCTCP (%)'] = topo_df['AU Error vs DCTCP (%)'].values
            fmt_dict.update({'NS3 DCTCP (ns)': '{:,.0f}', 'G2 Err DCTCP (%)': '{:+.2f}%', 'AU Err DCTCP (%)': '{:+.2f}%'})
        if has_dcqcn:
            summary_dict['NS3 DCQCN (ns)']   = topo_df['NS3_DCQCN_Est_Exec_Time_ns'].values
            summary_dict['G2 Err DCQCN (%)'] = topo_df['G2 Error vs DCQCN (%)'].values
            summary_dict['AU Err DCQCN (%)'] = topo_df['AU Error vs DCQCN (%)'].values
            fmt_dict.update({'NS3 DCQCN (ns)': '{:,.0f}', 'G2 Err DCQCN (%)': '{:+.2f}%', 'AU Err DCQCN (%)': '{:+.2f}%'})
        summary_dict['G2 (ns)'] = topo_df['G2_Est_Exec_Time_ns'].values
        summary_dict['AU (ns)'] = topo_df['Analytical_Est_Exec_Time_ns'].values
        fmt_dict.update({'G2 (ns)': '{:,.0f}', 'AU (ns)': '{:,.0f}'})

        summary_df = pd.DataFrame(summary_dict)
        print(f"\n--- Summary Table for {topo} ---")
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 320):
            display(summary_df.style.format(fmt_dict))

        # ---------------------------------------------------------------
        # LaTeX Table 1: Estimated Execution Time (one table per GT)
        # ---------------------------------------------------------------
        gt_variants = []
        if has_dctcp: gt_variants.append(('DCTCP', 'NS3_DCTCP_Est_Exec_Time_ns', 'G2 Error vs DCTCP (%)', 'AU Error vs DCTCP (%)'))
        if has_dcqcn: gt_variants.append(('DCQCN', 'NS3_DCQCN_Est_Exec_Time_ns', 'G2 Error vs DCQCN (%)', 'AU Error vs DCQCN (%)'))

        for gt_name, ns3_col, g2_err_col, au_err_col in gt_variants:
            est_df = pd.DataFrame({
                'Workload':            topo_df['short_workload'].values,
                f'NS3 {gt_name} (s)': (topo_df[ns3_col] / 1e9).values,
                'G2 (s)':             (topo_df['G2_Est_Exec_Time_ns'] / 1e9).values,
                'G2 Err. (%)':        topo_df[g2_err_col].values,
                'Analytic (s)':       (topo_df['Analytical_Est_Exec_Time_ns'] / 1e9).values,
                'Analytic Err. (%)':  topo_df[au_err_col].values,
            })
            avg_row = pd.DataFrame([{
                'Workload':            '\\textbf{Average / MAPE}',
                f'NS3 {gt_name} (s)': est_df[f'NS3 {gt_name} (s)'].mean(),
                'G2 (s)':             est_df['G2 (s)'].mean(),
                'G2 Err. (%)':        est_df['G2 Err. (%)'].abs().mean(),
                'Analytic (s)':       est_df['Analytic (s)'].mean(),
                'Analytic Err. (%)':  est_df['Analytic Err. (%)'].abs().mean(),
            }])
            est_df = pd.concat([est_df, avg_row], ignore_index=True)
            est_latex = est_df.to_latex(
                index=False, escape=False,
                formatters={
                    f'NS3 {gt_name} (s)': "{:.3f}".format,
                    'G2 (s)':             "{:.3f}".format,
                    'G2 Err. (%)':        "{:+.2f}\\%".format,
                    'Analytic (s)':       "{:.3f}".format,
                    'Analytic Err. (%)':  "{:+.2f}\\%".format,
                },
                caption=f'Estimated Execution Times for {topo} (GT: NS3 {gt_name}).',
                label=f'tab:est_times_{topo.lower()}_{gt_name.lower()}',
                position='!htbp',
                column_format='lccccc',
            )
            lines = est_latex.splitlines()
            lines.insert(-2, '\\hline')
            est_latex = '\n'.join(lines).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
            print(f"\n--- LaTeX: Estimated Execution Time for {topo} (GT: NS3 {gt_name}) ---")
            print(est_latex)

        # ---------------------------------------------------------------
        # LaTeX Table 2: Simulation Time & Speedup
        # ---------------------------------------------------------------
        sim_dict = {
            'Workload': topo_df['short_workload'].values,
            'G2 (s)':   topo_df['G2_Sim_Time_sec'].values,
            'Analytic (s)': topo_df['Analytical_Sim_Time_sec'].values,
        }
        sim_fmt = {'G2 (s)': "{:.3f}".format, 'Analytic (s)': "{:.3f}".format}
        avg_sim = {'Workload': '\\textbf{Average}', 'G2 (s)': topo_df['G2_Sim_Time_sec'].mean(), 'Analytic (s)': topo_df['Analytical_Sim_Time_sec'].mean()}

        if has_dctcp:
            sim_dict['NS3 DCTCP (s)']  = topo_df['NS3_DCTCP_Sim_Time_sec'].values
            sim_dict['G2 Spdup DCTCP'] = topo_df['G2 Speedup vs DCTCP (x)'].values
            sim_dict['AU Spdup DCTCP'] = topo_df['AU Speedup vs DCTCP (x)'].values
            sim_fmt.update({'NS3 DCTCP (s)': "{:.3f}".format, 'G2 Spdup DCTCP': "{:.2f}x".format, 'AU Spdup DCTCP': "{:.2f}x".format})
            avg_sim.update({'NS3 DCTCP (s)': topo_df['NS3_DCTCP_Sim_Time_sec'].mean(),
                            'G2 Spdup DCTCP': topo_df['G2 Speedup vs DCTCP (x)'].mean(),
                            'AU Spdup DCTCP': topo_df['AU Speedup vs DCTCP (x)'].mean()})
        if has_dcqcn:
            sim_dict['NS3 DCQCN (s)']  = topo_df['NS3_DCQCN_Sim_Time_sec'].values
            sim_dict['G2 Spdup DCQCN'] = topo_df['G2 Speedup vs DCQCN (x)'].values
            sim_dict['AU Spdup DCQCN'] = topo_df['AU Speedup vs DCQCN (x)'].values
            sim_fmt.update({'NS3 DCQCN (s)': "{:.3f}".format, 'G2 Spdup DCQCN': "{:.2f}x".format, 'AU Spdup DCQCN': "{:.2f}x".format})
            avg_sim.update({'NS3 DCQCN (s)': topo_df['NS3_DCQCN_Sim_Time_sec'].mean(),
                            'G2 Spdup DCQCN': topo_df['G2 Speedup vs DCQCN (x)'].mean(),
                            'AU Spdup DCQCN': topo_df['AU Speedup vs DCQCN (x)'].mean()})

        sim_df = pd.DataFrame(sim_dict)
        sim_df = pd.concat([sim_df, pd.DataFrame([avg_sim])], ignore_index=True)
        col_fmt = 'l' + 'c' * (len(sim_df.columns) - 1)
        sim_latex = sim_df.to_latex(index=False, escape=False, formatters=sim_fmt,
                                    caption=f'Simulation Times and Speedup for {topo} Topology.',
                                    label=f'tab:sim_times_{topo.lower()}',
                                    position='!htbp', column_format=col_fmt)
        lines = sim_latex.splitlines()
        lines.insert(-2, '\\hline')
        sim_latex = '\n'.join(lines).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
        print(f"\n--- LaTeX: Simulation Time for {topo} ---")
        print(sim_latex)

    # ---------------------------------------------------------------
    # Overall MAPE summary
    # ---------------------------------------------------------------
    print(f"\n{'='*80}\n--- Overall Average Absolute Error (MAPE) ---\n{'='*80}")
    if has_dctcp:
        print(f"G2 MAPE vs NS3 DCTCP:         {df['G2 Error vs DCTCP (%)'].abs().mean():.2f}%")
        print(f"Analytical MAPE vs NS3 DCTCP: {df['AU Error vs DCTCP (%)'].abs().mean():.2f}%")
    if has_dcqcn:
        print(f"G2 MAPE vs NS3 DCQCN:         {df['G2 Error vs DCQCN (%)'].abs().mean():.2f}%")
        print(f"Analytical MAPE vs NS3 DCQCN: {df['AU Error vs DCQCN (%)'].abs().mean():.2f}%")

else:
    print("No results were found to process.")
